In [1]:
import json
import random
import re
import tqdm
from collections import defaultdict

## 1.Read Predicted Data and Ground Truth Data

In [3]:
# read the ground truth data
Table_Evol_eval_test_data = json.load(open("path_to_the_original_test_data/TableDreamer_test_data_30K.json"))
# item_id --> test data
item_id_to_test_item = {}
for item in Table_Evol_eval_test_data:
    item_id = item['item_id']
    item_id_to_test_item[item_id] = item
print("Table-Evol test data num: ",len(item_id_to_test_item))


Table-Evol test data num:  30472


In [3]:
Table_Evol_eval_test_data[0]

{'item_id': 'TABMWP_8',
 'benchmark': 'TABMWP',
 'answer_list': ['18'],
 'table_id': 'TABMWP_8',
 'table_format': 'HTML',
 'table_title': 'Cookies baked',
 'input_text': 'Problem: \nHannah baked cookies each day for a bake sale. How many more cookies did Hannah bake on Saturday than on Sunday? (Unit: cookies)\nSolve the above problem based on the table titled \'Cookies baked\'. Provide a concluding answer in a JSON structure, using the format {"answer": "<YOUR ANSWER>"}.\n\nTable:\n<table border="1" cellspacing="0">\n<tr> <th colspan="2"> Cookies baked </th> </tr><tr> <th> Day </th> <th> Number of cookies </th> </tr>\n<tr> <td> Friday </td> <td> 163 </td> </tr>\n<tr> <td> Saturday </td> <td> 281 </td> </tr>\n<tr> <td> Sunday </td> <td> 263 </td> </tr>\n</table>\n\nResponse:\n'}

In [4]:
# Read model prediction data.
# The prediction data format looks like:
"""
{
    'item_id': 'WTQ_nu-0',
    'input': 'Please provide your detailed answer to the question below......',
    'model_output': 'To determine which country had the most cyclists finish......'
}
"""
predicted_item_list = []
with open('path_to_model_prediction/TableDreamer_pred_results.jsonl') as f:
    for line in f:
        item = json.loads(line.strip())
        item_id = item['item_id']
        if item_id in item_id_to_test_item:
            predicted_item_list.append(item)
print('Prediction sample num:',len(predicted_item_list))
# benchmark_name --> item_list
benchmark_name_to_predicted_item_list = defaultdict(list)
for item in predicted_item_list:
    item_id = item['item_id']
    test_item = item_id_to_test_item[item_id]
    benchmark_name = test_item['benchmark']
    benchmark_name_to_predicted_item_list[benchmark_name].append(item)
for benchmark_name,  item_list in benchmark_name_to_predicted_item_list.items():
        item_num = len(item_list)
        print(f'benchmark name: {benchmark_name}, test data num: {item_num}')

Prediction sample num: 30472
benchmark name: TABMWP, test data num: 7686
benchmark name: WTQ, test data num: 4344
benchmark name: FeTaQA, test data num: 2003
benchmark name: HiTab, test data num: 1576
benchmark name: TabFact, test data num: 6845
benchmark name: TabMCQ, test data num: 1029
benchmark name: AIT-QA, test data num: 511
benchmark name: InfoTabs, test data num: 5400
benchmark name: QTSumm, test data num: 1078


In [5]:
benchmark_name_to_predicted_item_list['WTQ'][0]

{'item_id': 'WTQ_nu-0',
 'input': 'Please provide your detailed answer to the question below based on the given table. In addition, output the final answer as JSON in the format {"answer": [<a list of answer strings>]}.\nwhich country had the most cyclists finish within the top 10?\n\nTable title: 2008 Clásica de San Sebastián\n\nTable:\nRank\tCyclist\tTeam\tTime\tUCI ProTour\\nPoints\n1\tAlejandro Valverde\xa0(ESP)\tCaisse d\'Epargne\t5h 29\' 10"\t40\n2\tAlexandr Kolobnev\xa0(RUS)\tTeam CSC Saxo Bank\ts.t.\t30\n3\tDavide Rebellin\xa0(ITA)\tGerolsteiner\ts.t.\t25\n4\tPaolo Bettini\xa0(ITA)\tQuick Step\ts.t.\t20\n5\tFranco Pellizotti\xa0(ITA)\tLiquigas\ts.t.\t15\n6\tDenis Menchov\xa0(RUS)\tRabobank\ts.t.\t11\n7\tSamuel Sánchez\xa0(ESP)\tEuskaltel-Euskadi\ts.t.\t7\n8\tStéphane Goubert\xa0(FRA)\tAg2r-La Mondiale\t+ 2"\t5\n9\tHaimar Zubeldia\xa0(ESP)\tEuskaltel-Euskadi\t+ 2"\t3\n10\tDavid Moncoutié\xa0(FRA)\tCofidis\t+ 2"\t1',
 'model_output': 'To determine which country had the most cycli

## 2.Evaluation Functions

In [5]:
def extract_tqa_answer_list(model_output):
    """
    Extract the answer list from the model output to compute accuracy
    """
    model_output = model_output.replace('\n',' ')
    ret = re.match('.*({\s*[\"\']answer[\"\']\:.*}).*',model_output)
    if ret is not None:
        answer_str = ret.group(1)
        try:
            answer_item = eval(answer_str)
            predicted_answer = answer_item['answer']
            if type(predicted_answer) != list and type(predicted_answer) == str:
                predicted_answer = [predicted_answer.rstrip('0%$')]
            elif type(predicted_answer) != list and type(predicted_answer) in [float,int]:
                predicted_answer = [str(predicted_answer)]
            else:
                pass
        except:
            predicted_answer = []
        return predicted_answer
    else:
        return []

def evaluate_tqa_questions(benchmark_name,pred_item_list):
    """
    Evaluation for table question answering (TQA) and table fact verification (TFV) benchmark.
    Metric: accuracy.
    Note that some baseline models can not strictly follow instructions to output the final answer in the required JSON format.
    In such cases, the evaluation script needs to be changed according to the characteristic of certain model output.
    """
    correct_item_list = []
    wrong_item_list = []
    failed_item_list = []
    for item in pred_item_list:
        try:
            item_id = item['item_id']
            ori_item = item_id_to_test_item[item_id]
            model_output = item['model_output'].lower()
            # parse the predicted answer list
            predicted_answer_list = extract_tqa_answer_list(model_output)
            gold_answer_list = [ answer_str.lower() for answer_str in ori_item['answer_list']]
            processed_gold_answer_list = [ answer_str.lower().rstrip('0%$') for answer_str in ori_item['answer_list']]
            # Sometimes the order of multiple answer text is not necessarily same as the gold answer,
            # so we convert the answer list to a set for comparison
            if predicted_answer_list != []:
                if set(predicted_answer_list) == set(gold_answer_list) or set(predicted_answer_list) == set(processed_gold_answer_list):
                    correct_item_list.append(item)
                else:
                    wrong_item_list.append(item)
            else:
                # we failed to extract the JSON answer, 
                # so we try to judge the correctness based on the potential short answer within the last 20 characters of the predicted model output.
                if len(ori_item['answer_list']) == 1 and ori_item['answer_list'][0].lower() in [model_output[-20:], model_output.strip('0%$')[-20:]]:
                    correct_item_list.append(item)
                elif ', '.join(ori_item['answer_list']) in model_output[-20:]:
                    correct_item_list.append(item)
                else:
                    wrong_item_list.append(item)
                failed_item_list.append(item)
            item['predicted_answer_list'] = predicted_answer_list
            item['gold_answer_list'] = gold_answer_list
        except Exception as e:
            item['exception'] = e
            failed_item_list.append(item)
    
    
    print("Benchmark: ",benchmark_name)
    correct_num = len(correct_item_list)
    total_sample_num = len(pred_item_list)
    
    print("Accuracy: ", round(correct_num*100/total_sample_num,5))
    problem_sample_num = len(failed_item_list)
    print('Correct sample number:',correct_num)
    print("Total sample number:",total_sample_num)
    print(f"There are {problem_sample_num} samples that failed to be evaluated with json output.")
    print("-"*20)
    return correct_item_list, wrong_item_list, failed_item_list


def evaluate_tabmcq_questions(benchmark_name,pred_item_list):
    """
    Evaluation for TabMCQ benchmark.
    Metric: accuracy. 
    """
    correct_item_list = []
    wrong_item_list = []
    failed_item_list = []
    
    for item in pred_item_list:
        try:
            item_id = item['item_id']
            ori_item = item_id_to_test_item[item_id]
            model_output = item['model_output'].lower()
            model_output = model_output.replace('\n',' ')
            model_output = model_output.replace(']','')
            model_output = model_output.replace('[','')
            ret = re.match('.*({\s*[\"\']answer[\"\']\:\s?.*?}).*',model_output)
            gold_answer_str = ori_item['answer_list'][0].lower() # e.g., '(D) Blood'
            # parse predicted JSON answer
            if ret is not None:
                answer_str = ret.group(1)
                answer_item = eval(answer_str)
                predicted_answer = str(answer_item['answer'])
                if type(predicted_answer) == list:
                    predicted_answer = predicted_answer[0]
                # Sometimes the predicted answer does not contain option letter like '(D)'
                # To deal with such cases, we also consider removing the option letter in ground truth for comparison
                correct_answer_option_str_list = [gold_answer_str.split(' ')[0], gold_answer_str.split(' ')[0].strip('().')] # '(a)', 'a'
                if predicted_answer == gold_answer_str or predicted_answer == ' '.join(gold_answer_str.split(' ')[1:]) or predicted_answer in correct_answer_option_str_list:
                    correct_item_list.append(item)
                else:
                    wrong_item_list.append(item)
            # trying to match the short answer
            elif gold_answer_str in [model_output[-20:]]:
                correct_item_list.append(item)
                failed_item_list.append(item)
            else:
                wrong_item_list.append(item)
                failed_item_list.append(item)
        except Exception as e:
            failed_item_list.append(item)
            item['exception'] = e
    
    print(f"Benchmark: {benchmark_name}")
    total_sample_num = len(pred_item_list)
    correct_num = len(correct_item_list)
    print("Accuracy: ",round(correct_num*100/total_sample_num,4))
    problem_sample_num = len(failed_item_list)
    print("Total sample number:",total_sample_num)
    print(f"There are {problem_sample_num} samples that failed to be evaluated with json output.")
    print("-"*20)
    return correct_item_list, wrong_item_list, failed_item_list


## 3.Evaluation of TQA and TFV Tasks (based on parsed JSON answer)

In [6]:
model_id = "Llama3.1-8B_with_TableDreamer_27K_GPT-4o"
benchmark_name_list = ['TABMWP','WTQ','HiTab','TabFact','InfoTabs','AIT-QA','TabMCQ']
benchmark_name_to_failed_item_list = defaultdict(dict)
benchmark_name_to_correct_item_list = defaultdict(dict)
benchmark_name_to_wrong_item_list = defaultdict(dict)
for benchmark_name in benchmark_name_list:
    predicted_item_list = benchmark_name_to_predicted_item_list[benchmark_name]
    
    if benchmark_name == 'TabMCQ':
        correct_item_list, wrong_item_list, failed_item_list = evaluate_tabmcq_questions(benchmark_name,predicted_item_list)
        benchmark_name_to_correct_item_list[model_id][benchmark_name] = correct_item_list
        benchmark_name_to_wrong_item_list[model_id][benchmark_name] = wrong_item_list
    else:
        correct_item_list, wrong_item_list, failed_item_list = evaluate_tqa_questions(benchmark_name,predicted_item_list)
        benchmark_name_to_correct_item_list[model_id][benchmark_name] = correct_item_list
        benchmark_name_to_wrong_item_list[model_id][benchmark_name] = wrong_item_list
    benchmark_name_to_failed_item_list[model_id][benchmark_name] = failed_item_list

Benchmark:  TABMWP
Accuracy:  64.62399
Correct sample number: 4967
Total sample number: 7686
There are 59 samples that failed to be evaluated with json output.
--------------------
Benchmark:  WTQ
Accuracy:  54.58103
Correct sample number: 2371
Total sample number: 4344
There are 85 samples that failed to be evaluated with json output.
--------------------
Benchmark:  HiTab
Accuracy:  23.2868
Correct sample number: 367
Total sample number: 1576
There are 37 samples that failed to be evaluated with json output.
--------------------
Benchmark:  TabFact
Accuracy:  63.02411
Correct sample number: 4314
Total sample number: 6845
There are 66 samples that failed to be evaluated with json output.
--------------------
Benchmark:  InfoTabs
Accuracy:  57.74074
Correct sample number: 3118
Total sample number: 5400
There are 23 samples that failed to be evaluated with json output.
--------------------
Benchmark:  AIT-QA
Accuracy:  53.22896
Correct sample number: 272
Total sample number: 511
There a

In [7]:
benchmark_name_to_predicted_item_list['WTQ'][0]

{'item_id': 'WTQ_nu-0',
 'input': 'Please provide your detailed answer to the question below based on the given table. In addition, output the final answer as JSON in the format {"answer": [<a list of answer strings>]}.\nwhich country had the most cyclists finish within the top 10?\n\nTable title: 2008 Clásica de San Sebastián\n\nTable:\nRank\tCyclist\tTeam\tTime\tUCI ProTour\\nPoints\n1\tAlejandro Valverde\xa0(ESP)\tCaisse d\'Epargne\t5h 29\' 10"\t40\n2\tAlexandr Kolobnev\xa0(RUS)\tTeam CSC Saxo Bank\ts.t.\t30\n3\tDavide Rebellin\xa0(ITA)\tGerolsteiner\ts.t.\t25\n4\tPaolo Bettini\xa0(ITA)\tQuick Step\ts.t.\t20\n5\tFranco Pellizotti\xa0(ITA)\tLiquigas\ts.t.\t15\n6\tDenis Menchov\xa0(RUS)\tRabobank\ts.t.\t11\n7\tSamuel Sánchez\xa0(ESP)\tEuskaltel-Euskadi\ts.t.\t7\n8\tStéphane Goubert\xa0(FRA)\tAg2r-La Mondiale\t+ 2"\t5\n9\tHaimar Zubeldia\xa0(ESP)\tEuskaltel-Euskadi\t+ 2"\t3\n10\tDavid Moncoutié\xa0(FRA)\tCofidis\t+ 2"\t1',
 'model_output': 'To determine which country had the most cycli

## 4.Evaluation of T2T Task (based on LLM-as-a-judge)

In [8]:
LLM_as_judge_template_for_T2T = """You will be given a prompt of a table-related task, a reference response and a response from a language model (LM).
Your task is to rate the correctness of the LM's response on a 5 point likert scale.
The Detailed Rating Breakdown is as follows.
- 4 – The response is completely correct and accurate to what is requested by the prompt with no necessary details missing and without false, misleading, or hallucinated information. If the prompt asks the LM to do a task, the task is completely done and addressed in the response.
- 3 – The response is mostly correct and accurate with a small amount of missing information. It contains no misleading information or hallucinations. If the prompt asks the LM to perform a task, the task is mostly successfully attempted.
- 2 – The response contains a mix of correct and incorrect information. The response may miss some information, contain misleading information, or minor hallucinations, but is more or less aligned with what the prompt asks for. If the prompt asks the LM to perform a task, the task is attempted with moderate success but still has clear room for improvement.
- 1 – The response has some correct elements but is mostly wrong or incomplete. The response may contain multiple instances of hallucinations, false information, misleading information, or irrelevant information. If the prompt asks the LM to do a task, the task was attempted with a small amount of success.
- 0 – The response is completely incorrect. All information provided is wrong, false or hallucinated. If the prompt asks the LM to do a task, the task is not at all attempted, or the wrong task was attempted in the response. The response is completely irrelevant to the prompt.

Format your output in the JSON format:
```json
{{
    'rating': <final rating result, string format>
    'explanation': <rating explanation, string format>
}}
```
##Input Prompt: ##
{}

##Reference Response: ##
{}

##Model Response to be rated: ##
{}

##Rating Result: ##
"""

# build LLM-as-a-judge input prompt for T2T Task samples (FeTaQA and QTSumm)
llm_judge_item_list = []
fetaqa_predict_results = benchmark_name_to_predicted_item_list['FeTaQA']
qtsumm_predict_results = benchmark_name_to_predicted_item_list['QTSumm']
for item in fetaqa_predict_results+qtsumm_predict_results:
    item_id = item['item_id']
    ori_test_item = item_id_to_test_item[item_id]
    benchmark_name = ori_test_item['benchmark']
    ground_truth = ori_test_item['answer_list'][0]
    input_prompt = item['input']
    pred_model_output = str(item['model_output']).strip()
    llm_as_judge_prompt = LLM_as_judge_template_for_T2T.format(input_prompt,ground_truth,pred_model_output)
    new_item = {}
    new_item['item_id'] = item_id
    new_item['benchmark_name'] = benchmark_name
    new_item['llm_as_judge_prompt'] = llm_as_judge_prompt # used for prompting LLM like GPT-4o
    llm_judge_item_list.append(new_item)

In [9]:
llm_judge_item_list[0]

{'item_id': 'FeTaQA_2206',
 'benchmark_name': 'FeTaQA',
 'llm_as_judge_prompt': "You will be given a prompt of a table-related task, a reference response and a response from a language model (LM).\nYour task is to rate the correctness of the LM's response on a 5 point likert scale.\nThe Detailed Rating Breakdown is as follows.\n- 4 – The response is completely correct and accurate to what is requested by the prompt with no necessary details missing and without false, misleading, or hallucinated information. If the prompt asks the LM to do a task, the task is completely done and addressed in the response.\n- 3 – The response is mostly correct and accurate with a small amount of missing information. It contains no misleading information or hallucinations. If the prompt asks the LM to perform a task, the task is mostly successfully attempted.\n- 2 – The response contains a mix of correct and incorrect information. The response may miss some information, contain misleading information, or 

In [ ]:
# Load the LLM-as-a-judge results after prompting the LLM
t2t_llm_judge_results = json.load(open('path_to_the_results/llm_as_a_judge_results_for_T2T_task.json'))

In [ ]:
# Parse the LLM-as-a-judge results after prompting the LLM
benchmark_name_to_llm_judge_results = defaultdict(list)
benchmark_name_to_failed_llm_judge_results = defaultdict(list)

for item in t2t_llm_judge_results:
    benchmark_name = item['benchmark_name']
    model_output = item['model_output']
    try:
        rating_score = re.findall(r"[\"\']rating[\"\']:\s*(.*),",model_output)[0].strip("\"\'")
        rating_explanation = re.findall(r"[\"\']explanation[\"\']:\s*(.*)",model_output)[0].strip("\"\'")
        item['result_item'] = {
            "rating_score": rating_score,
            "rating_explanation": rating_explanation
        }
    except:
        rating_score = '0'
        rating_explanation = 'parsing failed'
        item['result_item'] = {
            "rating_score": rating_score,
            "rating_explanation": rating_explanation
        }
        benchmark_name_to_failed_llm_judge_results[benchmark_name].append(item)
    finally:
        benchmark_name_to_llm_judge_results[benchmark_name].append(item)

print('Total sample number:')
for benchmark_name, item_list in benchmark_name_to_llm_judge_results.items():
    print(benchmark_name,len(item_list))
print('Number of samples that we failed to parse the JSON results:')
for benchmark_name, item_list in benchmark_name_to_failed_llm_judge_results.items():
        print(benchmark_name,len(item_list))
# computing the accuracy based on the LLM-as-a-judge results
print('Accuracy: ')
for benchmark_name, item_list in benchmark_name_to_llm_judge_results.items():
    correct_num = 0
    for item in item_list:
        rating_score = item['result_item']['rating_score']
        # only samples with rating score of 4 are considered as correct samples.
        if rating_score == '4':
            correct_num += 1
    print(benchmark_name,f'Correct sample Number: {correct_num}, Acc:',round(correct_num/len(item_list),5))

## 5.Evaluation of TableGPT benchmark

Refer to the offical github for test data and evaluation scripts of TableGPT dataset.
https://github.com/microsoft/Table-GPT